# FloodLens accuracy check
Runs FloodLens's exact prompt (read from `../src/app.py`) on photos where you know the true water depth, then reports how close it gets.

**Before running**
1. Put photos in `notebooks/eval_photos/`.
2. Fill `notebooks/ground_truth.csv` (columns `filename,true_depth_cm`). Get true depth by measuring the water with a tape or ruler at the moment of the photo, with a known object in frame.
3. Have AWS credentials with Bedrock access, and set `MODEL_ID` to the model you deployed with.

The numbers are only as good as your test set, so report how many photos you used.

In [ ]:
%pip install -q boto3 pandas matplotlib pillow

In [ ]:
import ast, io, json, os, pathlib, boto3, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
REGION = os.environ.get("AWS_REGION", "ap-south-1")
MODEL_ID = os.environ.get("MODEL_ID", "apac.anthropic.claude-sonnet-4-20250514-v1:0")  # use the ID you deployed with
PHOTOS, TRUTH = pathlib.Path("eval_photos"), pathlib.Path("ground_truth.csv")
tree = ast.parse(pathlib.Path("../src/app.py").read_text())
keep = [n for n in tree.body if (isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in ("LIMITS", "RANK", "SYSTEM")) or (isinstance(n, ast.FunctionDef) and n.name == "verdicts")]
ns = {}; exec(compile(ast.Module(keep, []), "app.py", "exec"), ns)
SYSTEM, verdicts, RANK = ns["SYSTEM"], ns["verdicts"], ns["RANK"]
br = boto3.client("bedrock-runtime", region_name=REGION)

In [ ]:
def jpeg(path):
    im = Image.open(path).convert("RGB"); im.thumbnail((1024, 1024))
    b = io.BytesIO(); im.save(b, "JPEG", quality=82); return b.getvalue()

def estimate(path):
    r = br.converse(modelId=MODEL_ID, system=[{"text": SYSTEM}],
        messages=[{"role": "user", "content": [{"image": {"format": "jpeg", "source": {"bytes": jpeg(path)}}}, {"text": "Estimate the flood depth. JSON only."}]}],
        inferenceConfig={"maxTokens": 700, "temperature": 0})
    t = r["output"]["message"]["content"][0]["text"]
    return json.loads(t[t.find("{"):t.rfind("}") + 1])

truth = pd.read_csv(TRUTH); rows = []
for _, g in truth.iterrows():
    p = PHOTOS / g.filename
    if not p.exists(): print("skipping missing", p); continue
    try: d = estimate(p)
    except Exception as e: print("failed", p, e); continue
    if d.get("is_flood") is not True: print("not detected as flood:", p); continue
    lo, hi = float(d["depth_low_cm"]), float(d["depth_high_cm"])
    rows.append(dict(file=g.filename, truth=float(g.true_depth_cm), low=lo, high=hi, mid=(lo + hi) / 2, conf=d.get("confidence")))
df = pd.DataFrame(rows); df

In [ ]:
worst = lambda h: max(verdicts(h).values(), key=RANK.get)
df["abs_err"] = (df.mid - df.truth).abs()
df["in_range"] = (df.truth >= df.low) & (df.truth <= df.high)
df["truth_verdict"], df["pred_verdict"] = df.truth.map(worst), df.high.map(worst)
df["unsafe_miss"] = df.apply(lambda r: RANK[r.truth_verdict] > RANK[r.pred_verdict], axis=1)
summary = {"photos in ground_truth.csv": len(truth), "photos evaluated": len(df),
  "mean abs error (cm)": round(df.abs_err.mean(), 1), "median abs error (cm)": round(df.abs_err.median(), 1),
  "measured depth inside predicted range (%)": round(100 * df.in_range.mean()),
  "worst-case verdict matches (%)": round(100 * (df.truth_verdict == df.pred_verdict).mean()),
  "unsafe misses (real verdict worse than predicted)": int(df.unsafe_miss.sum())}
pd.Series(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.errorbar(df.truth, df.mid, yerr=[df.mid - df.low, df.high - df.mid], fmt="o", capsize=3)
m = max(df.truth.max(), df.high.max()) + 10; ax.plot([0, m], [0, m], "--", color="gray")
ax.set_xlabel("Measured depth (cm)"); ax.set_ylabel("FloodLens estimate (cm)"); ax.set_title("FloodLens vs measured depth")
fig.savefig("eval_plot.png", dpi=150, bbox_inches="tight"); plt.show()

## Use these results
Copy the summary table into the **Validation** section of your project post, with the number of photos and how you measured true depth. Pay attention to *unsafe misses*: a tool that underestimates dangerous water is worse than one that overestimates. Report that number even if it is not zero.